<a href="https://colab.research.google.com/github/LeJ-04/web-datamining-semantics/blob/kb-modeling/notebooks/kb_modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab Session: Knowledge Base Construction,
Alignment, and Expansion

# Step 1 --- Build the Initial Private Knowledge Base

Libraries

In [ ]:
%pip install -q rdflib SPARQLWrapper

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 10.2 MB/s eta 0:00:00


In [ ]:
import os
import time
import requests
import numpy as np
import pandas as pd

from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF, RDFS, OWL

from SPARQLWrapper import SPARQLWrapper, JSON

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
project_path = "/content/drive/MyDrive/Home/School/ESILV/A4-DIA/Semestre 2/Web Datamining & Semantics/TP4"

%cd "{project_path}"

/content/drive/MyDrive/Home/School/ESILV/A4-DIA/Semestre 2/Web Datamining & Semantics/TP4


In [ ]:
from google.colab import userdata

In [ ]:
os.environ["FOOTBALL_DATA_API_KEY"] = userdata.get('FOOTBALL_DATA_API_KEY')

API Key

In [ ]:
API_KEY = os.getenv("FOOTBALL_DATA_API_KEY")
headers = {"X-Auth-Token": API_KEY}

In [ ]:
url = "https://api.football-data.org/v4/competitions/PL/teams"

response = requests.get(url, headers=headers)
data = response.json()

In [ ]:
display(data.keys())

dict_keys(['count', 'filters', 'competition', 'season', 'teams'])

In [ ]:
teams = data["teams"]
teams[0].keys()

dict_keys(['area', 'id', 'name', 'shortName', 'tla', 'crest', 'address', 'website', 'founded', 'clubColors', 'venue', 'runningCompetitions', 'coach', 'squad', 'staff', 'lastUpdated'])

In [ ]:
teams[0]["name"]

'Arsenal FC'

In [ ]:
teams[0]['area']

{'id': 2072,
 'name': 'England',
 'code': 'ENG',
 'flag': 'https://crests.football-data.org/770.svg'}

In [ ]:
teams[0]['runningCompetitions'][:5]

[{'id': 2021,
  'name': 'Premier League',
  'code': 'PL',
  'type': 'LEAGUE',
  'emblem': 'https://crests.football-data.org/PL.png'},
 {'id': 2001,
  'name': 'UEFA Champions League',
  'code': 'CL',
  'type': 'CUP',
  'emblem': 'https://crests.football-data.org/CL.png'}]

In [ ]:
teams[0]['coach']

{'id': 179744,
 'firstName': '',
 'lastName': 'Mikel Arteta',
 'name': 'Mikel Arteta',
 'dateOfBirth': '1982-03-26',
 'nationality': 'Spain',
 'contract': {'start': '2019-12', 'until': '2027-06'}}

In [ ]:
teams[10]['squad'][:5]

[{'id': 3086,
  'name': 'Guglielmo Vicario',
  'position': 'Goalkeeper',
  'dateOfBirth': '1996-10-07',
  'nationality': 'Italy'},
 {'id': 133229,
  'name': 'Brandon Austin',
  'position': 'Goalkeeper',
  'dateOfBirth': '1999-01-08',
  'nationality': 'England'},
 {'id': 255239,
  'name': 'Antonín Kinský',
  'position': 'Goalkeeper',
  'dateOfBirth': '2003-03-13',
  'nationality': 'Czech Republic'},
 {'id': 274554,
  'name': 'Luca Gunter',
  'position': 'Goalkeeper',
  'dateOfBirth': '2005-03-23',
  'nationality': 'England'},
 {'id': 249465,
  'name': 'Souza',
  'position': 'Defence',
  'dateOfBirth': '2006-06-16',
  'nationality': 'Brazil'}]

In [ ]:
for t in teams:
    print(t["name"], "@", t["area"]["name"])

Arsenal FC @ England
Aston Villa FC @ England
Chelsea FC @ England
Everton FC @ England
Fulham FC @ England
Liverpool FC @ England
Manchester City FC @ England
Manchester United FC @ England
Newcastle United FC @ England
Sunderland AFC @ England
Tottenham Hotspur FC @ England
Wolverhampton Wanderers FC @ England
Burnley FC @ England
Leeds United FC @ England
Nottingham Forest FC @ England
Crystal Palace FC @ England
Brighton & Hove Albion FC @ England
Brentford FC @ England
West Ham United FC @ England
AFC Bournemouth @ England


In [ ]:
len(teams)

20

In [ ]:
g = Graph()

EX = Namespace("http://www.example.org/football/")
g.bind("ex", EX)

EX_PROP = Namespace(EX + "prop/")
g.bind("prop", EX_PROP)

In [ ]:
for t in teams:

    team_uri = URIRef(EX + t["name"].replace(" ", "_"))
    g.add((team_uri, RDF.type, EX.Team))

    # Team's country
    team_country_uri = URIRef(EX + t["area"]["name"].replace(" ", "_"))
    g.add((team_country_uri, RDF.type, EX.Country))
    g.add((team_uri, EX_PROP.locatedIn, team_country_uri))


    # Team's coach
    coach_uri = URIRef(EX + t["coach"]["name"].replace(" ", "_"))
    g.add((coach_uri, RDF.type, EX.Person))
    g.add((team_uri, EX_PROP.headCoach, coach_uri))


    # Team's coach nationality
    coach_country_uri = URIRef(EX + t["coach"]["nationality"].replace(" ", "_"))
    g.add((coach_country_uri, RDF.type, EX.Country))
    g.add((coach_uri, EX_PROP.nationality, coach_country_uri))

In [ ]:
for t in teams:
    team_uri = URIRef(EX + t["name"].replace(" ", "_"))
    for player in t["squad"]:
        # Player of a team
        player_uri = URIRef(EX + player["name"].replace(" ", "_"))
        g.add((player_uri, RDF.type, EX.Person))
        g.add((player_uri, EX_PROP.playsFor, team_uri))


        # Player's nationality
        player_country_uri = URIRef(EX + player["nationality"].replace(" ", "_"))
        g.add((player_country_uri, RDF.type, EX.Country))
        g.add((player_uri, EX_PROP.nationality, player_country_uri))


        # Player's Position
        if player["position"]:
            position_uri = URIRef(EX + player["position"].replace(" ", "_"))
            g.add((position_uri, RDF.type, EX.FootballPosition))
            g.add((player_uri, EX_PROP.playsPosition, position_uri))

In [ ]:
print("Triplets Count:", len(g))

Triplets Count: 2772


In [ ]:
def get_graph_stats(graph):
    graph_stats = dict()
    graph_stats["num_triples"] = len(graph)

    triplets_stats_query = """
    SELECT (COUNT(DISTINCT ?s) AS ?num_subjects) (COUNT(DISTINCT ?p) AS ?num_predicates) (COUNT(DISTINCT ?o) AS ?num_objects)
    WHERE {
        ?s ?p ?o
    }
    """.strip()
    triplets_stats_query_result = next(iter(graph.query(triplets_stats_query)))
    graph_stats["num_subjects"] = triplets_stats_query_result.num_subjects
    graph_stats["num_objects"] = triplets_stats_query_result.num_objects
    graph_stats["num_predicates"] = triplets_stats_query_result.num_predicates

    num_entities_query = """
    SELECT (COUNT(DISTINCT ?entity) AS ?num)
    WHERE {
        {
            ?entity ?p ?o .
            FILTER(isIRI(?entity))
        }
        UNION
        {
            ?s ?p ?entity .
            FILTER(isIRI(?entity))
        }
    }
    """.strip()
    graph_stats["num_entities"] = next(iter(graph.query(num_entities_query))).num

    return graph_stats


def display_graph_stats(graph):
    graph_stats = get_graph_stats(graph)

    for k, v in graph_stats.items():
        var_name = k.split("_")[-1]
        if k.startswith("num_"):
            print(f"Number of {var_name}: {v}")

In [ ]:
display_graph_stats(g)

Number of triples: 2772
Number of subjects: 771
Number of objects: 128
Number of predicates: 6
Number of entities: 775


In [228]:
kb_fpath = "football_kb.ttl"

In [ ]:
g.serialize(kb_fpath, format="turtle")

<Graph identifier=Nc6a84a68735e4255809065c703f93a94 (<class 'rdflib.graph.Graph'>)>

# Step 2 --- Entity Linking with Open Knowledge Bases

In [ ]:
g = Graph().parse(kb_fpath)

len(g)

2772

Alignement

In [ ]:
import re
from difflib import SequenceMatcher

Confidence score based on textual similarity

In [ ]:
def normalize_str(s):
    s1 = re.sub('(.)([A-Z][a-z]+)', r'\1 \2', s)
    s1 = re.sub('([a-z0-9])([A-Z])', r'\1 \2', s1)

    s1 = s1.replace('_', ' ').replace('-', ' ')
    return " ".join(s1.lower().split())

def similarity_matcher_score(entity_name, retrieved_name):
    n1 = normalize_str(entity_name)
    n2 = normalize_str(retrieved_name)

    score = SequenceMatcher(None, n1, n2).ratio() * 100
    return round(score, 2)

Execute a sparql query over wikidata

In [ ]:
def single_wikidata_sparql(query):
    url = "https://query.wikidata.org/sparql"

    headers = {
        "User-Agent": "PL Football KGE [1.0]",
        "Accept": "application/sparql-results+json"
    }

    params = {
        "query": query,
        "format": "json"
    }
    r = requests.get(url, headers=headers, params=params)
    r.raise_for_status()
    data = r.json()
    return data

def exec_wikidata_sparql(query, max_retries=3):
    retries = 0
    while retries < max_retries:
        try:
            return single_wikidata_sparql(query)
        except requests.RequestException as e:
            retries += 1
            if retries == max_retries:
                print(f"Fails after {max_retries} attempts : {e}")
                raise

            # Exponential waiting time
            wait_time = 1.5 * retries
            print(f"An error occured. New attempt ({retries}/{max_retries}) in {wait_time}s...")
            time.sleep(wait_time)

In [ ]:
from urllib.error import HTTPError as urllib_HTTPError

def single_wikidata_sparql(query):
    url = "https://query.wikidata.org/sparql"

    wikidata_wrapper = SPARQLWrapper(url)
    wikidata_wrapper.setReturnFormat(JSON)

    wikidata_wrapper.setQuery(query)
    results = wikidata_wrapper.query().convert()

    return results


def exec_wikidata_sparql(query, max_retries=3):
    retries = 0
    while retries < max_retries:
        try:
            return single_wikidata_sparql(query)
        except urllib_HTTPError as e:
            retries += 1
            if retries == max_retries:
                print(f"Fails after {max_retries} attempts : {e}")
                raise

            # Exponential waiting time
            wait_time = 1.5 * retries
            print(f"Error {e.code}. New attempt ({retries}/{max_retries}) in {wait_time}s...")
            time.sleep(wait_time)

Search entity on wikidata

In [ ]:
def search_wikidata_entity(wd_id):
    query = f"""
    SELECT ?item ?itemLabel ?itemDescription ?itemType WHERE {{
      BIND(wd:{wd_id} AS ?item) .
      OPTIONAL {{
        ?item wdt:P31 ?itemType
      }}
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en,fr". }}
    }}
    """
    data = exec_wikidata_sparql(query)
    if data is None or not data.get("results") or not data["results"].get("bindings"):
        return None
    return data["results"]["bindings"][0]

In [ ]:
search_wikidata_entity("Q5") # human

{'item': {'type': 'uri', 'value': 'http://www.wikidata.org/entity/Q5'},
 'itemType': {'type': 'uri',
  'value': 'http://www.wikidata.org/entity/Q55983715'},
 'itemLabel': {'xml:lang': 'en', 'type': 'literal', 'value': 'human'},
 'itemDescription': {'xml:lang': 'en',
  'type': 'literal',
  'value': 'any single member of Homo sapiens, unique extant species of the genus Homo'}}

Search aligned entities on Wikidata for private entities

In [ ]:
def search_wikidata_aligned_entities_sparql(entity_name, constraints="", confidence_fn=None, k=1):
    """
    Search for an entity on Wikidata using SPARQL with text search and filters.

    Args:
        entity_name: The name to search for (e.g., “Arsenal”)
        constraints: The SPARQL constraint (e.g., “?item wdt:P31 wd:Q476028” for a club)
        k: Maximum number of results (LIMIT)
    """
    query = f"""
    SELECT ?item ?itemLabel ?itemDescription WHERE {{
      SERVICE wikibase:mwapi {{
          bd:serviceParam wikibase:api "EntitySearch" .
          bd:serviceParam wikibase:endpoint "www.wikidata.org" .
          bd:serviceParam mwapi:search "{entity_name}" .
          bd:serviceParam mwapi:language "en" .
          ?item wikibase:apiOutputItem mwapi:item .
      }}

      # Apply constraints
      {constraints} .

      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en,fr". }}
    }}
    LIMIT {k}
    """

    data = exec_wikidata_sparql(query)

    if data is None or not data.get("results") or not data["results"].get("bindings"):
        return None

    # Get top k search results
    top_k_results = data["results"]["bindings"]

    aligned_entities = []
    for result in top_k_results:
        # Prevent case where key exists but is set to None
        result_id = result["item"]["value"].split("/")[-1]
        result_label = result.get("itemLabel", {}).get("value", "N/A")
        result_desc = result.get("itemDescription", {}).get("value", "No description")

        # Compute confidence score
        score = np.nan
        if confidence_fn is not None:
            score = confidence_fn(entity_name, result_label)

        # Append aligned entity
        aligned_entities.append((result_id, result_label, score, result_desc))

    # Create dataframe about aligned entities
    result_df = pd.DataFrame(aligned_entities, columns=["id", "label", "score", "description"])
    # Remove columns with not any single value (score if confidence_fn param is set to None)
    return result_df.dropna(axis=1, how='all')

Get a team roster

In [ ]:
def get_wikidata_team_roster(team_wd_id):
    """
    Retrieves all players from a given team in a single SPARQL query.
    """

    query = f"""
    SELECT ?player ?playerLabel ?playerDescription WHERE {{
      ?player wdt:P31 wd:Q5 .          # is a human
      ?player wdt:P54 wd:{team_wd_id} . # plays for team (P54 = member of sports team)
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en,fr". }}
    }}
    """
    data = exec_wikidata_sparql(query)

    if data is None or not data.get("results") or not data["results"].get("bindings"):
        return None

    roster = []
    for result in data["results"]["bindings"]:
        p_id = result["player"]["value"].split("/")[-1]
        p_label = result.get("playerLabel", {}).get("value", "N/A")
        p_desc = result.get("playerDescription", {}).get("value", "")
        roster.append((p_id, p_label, p_desc))

    return pd.DataFrame(roster, columns=["id", "label", "description"])

Get all players of a team in private KB

In [ ]:
def extract_team_players(graph, team_uri):
    team_players = []

    for player in graph.subjects(EX_PROP.playsFor, team_uri):
        team_players.append(player)

    return team_players

Get entity type's ID

In [ ]:
TYPE_TO_WIKI_QUERY = {
    "Team": "Football Club",
    "Country": "Countries",
}

Search the best entity associated to a certain private type

In [ ]:
search_wikidata_aligned_entities_sparql(TYPE_TO_WIKI_QUERY["Team"], k=3)

,id,label,description
0,Q476028,association football club,sports club devoted to association football (s...
1,Q17374546,Australian rules football club,team which plays Australian football
2,Q17270000,football club,organization which fields teams in a sport kno...


In [ ]:
def type_to_query(type_uri):
    # Type name
    type_name = type_uri.split("/")[-1]

    # Query for type
    type_query = TYPE_TO_WIKI_QUERY.get(type_name)

    # Use type name if no query is found
    type_query = type_query or type_name.replace("_", " ")

    return type_query

In [ ]:
def get_entity_type_wikidata_id(entity_type_uri, confidence_fn=None):
    query = type_to_query(entity_type_uri)

    results = search_wikidata_aligned_entities_sparql(query, confidence_fn=confidence_fn)

    if results is None or results.empty:
        return None

    # Sort if there is a score
    if confidence_fn is not None:
        results = results.sort_values("score", ascending=False)

    best = results.iloc[0]
    return best["id"]

In [ ]:
get_entity_type_wikidata_id(EX.Team)

'Q476028'

In [ ]:
get_entity_type_wikidata_id(EX.Country)

'Q6256'

Search Arsenal FC with the constraint to get an entity of type wd:Q476028 (association football club)

In [ ]:
constraints = "?item wdt:P31 wd:Q476028"
search_wikidata_aligned_entities_sparql("Arsenal FC", constraints, similarity_matcher_score)

,id,label,score,description
0,Q9617,Arsenal F.C.,90.91,"association football club in London, England"


Extract all entities for a further alignement

In [ ]:
# def extract_entities_with_types(graph):

#     entities = {}

#     for s, p, o in graph.triples((None, RDF.type, None)):
#         if s not in entities:
#             entities[s] = set()
#         entities[s].add(o)

#     return entities

# list(extract_entities_with_types(g).items())[:5]

Align entities

In [ ]:
WIKIDATA_BASE = Namespace("http://www.wikidata.org/")
WIKIDATA_NS = Namespace(WIKIDATA_BASE + "entity/")

In [ ]:
def get_entity_best_alignement_from_wikidata(graph, entity_uri, entity_type_uri=None, confidence_fn=None):
    # Default function for confidence score
    if confidence_fn is None:
        confidence_fn = similarity_matcher_score

    # Obtain entity name
    entity_name = entity_uri.split("/")[-1].replace("_", " ")

    # Entity type's equivalent on wikidata
    entity_wd_type_id = None
    if entity_type_uri is not None:
        entity_wd_type_id = get_entity_type_wikidata_id(entity_type_uri)

    # Constraints on type
    constraints = ""
    if entity_wd_type_id is not None:
        constraints = f"?item wdt:P31 wd:{entity_wd_type_id}"

    # Search entity's equivalent on wikidata (with constraint)
    results = search_wikidata_aligned_entities_sparql(entity_name, constraints=constraints, confidence_fn=confidence_fn)

    # No match found
    if results is None or results.empty:
        return None

    # Best match
    best = results.sort_values("score", ascending=False).iloc[0]
    return best


In [ ]:
def align_team_players_with_wikidata(graph, team_uri, team_wd_uri, confidence_fn=None):
    score_threshold = 90

    completion_graph = Graph()
    aligned = {}
    unaligned = []

    wd_team_roster_df = None
    if team_wd_uri is not None:
        # Get team's players from wikidata
        wd_team_roster_df = get_wikidata_team_roster(team_wd_uri.split("/")[-1])

    # If team's players has been found from Wikidata
    if wd_team_roster_df is not None and not wd_team_roster_df.empty:
        # Evaluate each local entity of team's players with wikidata's players
        for player_uri in extract_team_players(graph, team_uri):
            player_name = player_uri.split("/")[-1].replace("_", " ")

            # Compute confidence score over all wikidata's players
            wd_team_roster_df["score"] = wd_team_roster_df["label"].apply(lambda wd_name: confidence_fn(player_name, wd_name))

            # Best match
            best_match = wd_team_roster_df.sort_values("score", ascending=False).iloc[0]

            # Add alignement if score is above threshold
            if best_match["score"] > score_threshold:
                player_wd_uri = URIRef(WIKIDATA_NS + best_match["id"]) # Player equivalent's Wikidata URI
                completion_graph.add((player_uri, OWL.sameAs, player_wd_uri))
                aligned[player_uri] = best_match.to_dict()
            else:
                unaligned.append(player_uri)

    # If some players has been aligned and some others have not
    if unaligned:
        remaining_team_players = unaligned
        unaligned = []

    # If not any single player has been aligned (specificly when wd_team_roster_df is None or empty)
    if not aligned:
        remaining_team_players = extract_team_players(graph, team_uri)

    # Align remaining players
    for player_uri in remaining_team_players:
        best_match = get_entity_best_alignement_from_wikidata(graph, player_uri, EX.Person, confidence_fn)
        if best_match is None:
            unaligned.append(player_uri)
            continue
        # Add alignement triplet
        player_wd_uri = URIRef(WIKIDATA_NS + best_match["id"]) # Player equivalent's Wikidata URI
        completion_graph.add((player_uri, OWL.sameAs, player_wd_uri))
        aligned[player_uri] = best_match.to_dict()

    return completion_graph, aligned, unaligned

In [ ]:
def align_teams_with_wikidata(graph, confidence_fn):
    completion_graph = Graph()
    aligned = {}
    unaligned = []

    for team_uri in graph.subjects(RDF.type, EX.Team):
        team_wd_uri = None

        best_match = get_entity_best_alignement_from_wikidata(graph, team_uri, EX.Team, confidence_fn)
        if best_match is not None:
            # Add alignement triplet
            team_wd_uri = URIRef(WIKIDATA_NS + best_match["id"]) # Team equivalent's Wikidata URI
            completion_graph.add((team_uri, OWL.sameAs, team_wd_uri))
            aligned[team_uri] = best_match.to_dict()
        else:
            unaligned.append(team_uri)

        # Align team players with wikidata
        p_completion_graph, p_aligned, p_unaligned = align_team_players_with_wikidata(graph, team_uri, team_wd_uri, confidence_fn)
        completion_graph += p_completion_graph
        aligned.update(p_aligned)
        unaligned.extend(p_unaligned)
        time.sleep(1.5)

    return completion_graph, aligned, unaligned

In [ ]:
def align_entities_with_wikidata(graph, confidence_fn):
    completion_graph, aligned, unaligned = align_teams_with_wikidata(graph, confidence_fn)

    get_remaining_entities_query = f"""
    PREFIX : <{EX}>
    PREFIX prop: <{EX_PROP}>
    PREFIX rdf: <{RDF}>

    SELECT DISTINCT ?s
    WHERE {{
        ?s ?p ?o .

        FILTER NOT EXISTS {{
            ?s rdf:type :Team .
        }}

        FILTER NOT EXISTS {{
            ?team rdf:type :Team .
            ?s prop:playsFor ?team .
        }}

    }}
    """.strip()

    for (entity_uri,) in g.query(get_remaining_entities_query):
        best_match = get_entity_best_alignement_from_wikidata(graph, entity_uri, None, confidence_fn)
        if best_match is None:
            unaligned.append(entity_uri)
            continue

        # Add alignement triplet
        entity_wd_uri = URIRef(WIKIDATA_NS + best_match["id"]) # Entity equivalent's Wikidata URI
        completion_graph.add((entity_uri, OWL.sameAs, entity_wd_uri))
        aligned[entity_uri] = best_match.to_dict()

    return completion_graph, aligned, unaligned

Define unaligned entities

In [ ]:
def define_unaligned_entities(graph, unaligned_entities, confidence_fn=None):
    completion_graph = Graph()

    # Unique types of unaligned entities
    unique_types = set()
    for entity_uri in unaligned_entities:
        for t in graph.objects(entity_uri, RDF.type):
            unique_types.add(t)

    for t in unique_types:
        t_wd_id = get_entity_type_wikidata_id(t)
        t_wd_uri = URIRef(WIKIDATA_NS + t_wd_id) # Type equivalent's Wikidata URI

        # Add definition triplet
        completion_graph.add((t, RDFS.subClassOf, t_wd_uri))

    return completion_graph

Start alignement

In [ ]:
len(g)

2772

In [ ]:
alignement_completion_graph, aligned, unaligned = align_entities_with_wikidata(g, similarity_matcher_score)

len(alignement_completion_graph)

733

Save alignement mappling table

In [ ]:
import csv

In [ ]:
alignement_mapping_table_fpath = "alignement_mapping_table.csv"

In [ ]:
with open(alignement_mapping_table_fpath,"w") as f:
    # Init CSV writer
    writer = csv.writer(f)
    # Header
    writer.writerow(["Private Entity", "External URI", "Confidence"])

    # Rows
    for entity, data in aligned.items():
        writer.writerow([entity, str(WIKIDATA_NS + data["id"]), data["score"]])

In [ ]:
alignement_mapping_df = pd.read_csv(alignement_mapping_table_fpath)

print("Shape:", alignement_mapping_df.shape)
alignement_mapping_df.head()

Shape: (733, 3)


,Private Entity,External URI,Confidence
0,http://www.example.org/football/Leeds_United_FC,http://www.wikidata.org/entity/Q1128631,93.75
1,http://www.example.org/football/Joe_Rodon,http://www.wikidata.org/entity/Q47658470,100.00
2,http://www.example.org/football/Alex_Cairns,http://www.wikidata.org/entity/Q4716779,100.00
3,http://www.example.org/football/Sam_Byram,http://www.wikidata.org/entity/Q5591335,100.00
4,http://www.example.org/football/Anton_Stach,http://www.wikidata.org/entity/Q66712655,100.00


Handle unaligned entities

In [ ]:
print("Unaligned Count:", len(unaligned), "\n")

unaligned[:5]

Unaligned Count: 38 



[rdflib.term.URIRef('http://www.example.org/football/Charlie_Crewe'),
 rdflib.term.URIRef('http://www.example.org/football/Jackson_Izquierdo'),
 rdflib.term.URIRef('http://www.example.org/football/George_Brierley'),
 rdflib.term.URIRef('http://www.example.org/football/Brandon_Pouani'),
 rdflib.term.URIRef('http://www.example.org/football/Kian_Brown')]

In [ ]:
definition_completion_graph = define_unaligned_entities(g, unaligned)

len(definition_completion_graph)

1

In [226]:
entities_aligned_graph_fpath = "ent_aligned_football_kb.ttl"

In [ ]:
entities_aligned_graph = g + alignement_completion_graph + definition_completion_graph

entities_aligned_graph.serialize(destination=entities_aligned_graph_fpath, format="turtle")

<Graph identifier=N5318709f2b79472888e9992862330f0b (<class 'rdflib.graph.Graph'>)>

In [ ]:
len(entities_aligned_graph)

3506

In [ ]:
# for uri in unaligned:
#     name = uri.split("/")[-1].replace("_", " ")

#     print(name, end=":")
#     display(search_wikidata_aligned_entities_sparql(name, k=5))
#     print()

# Step 3 --- Predicate Alignment Using SPARQL

In [233]:
g = Graph().parse(entities_aligned_graph_fpath)

In [234]:
display_graph_stats(g)

Number of triples: 3506
Number of subjects: 772
Number of objects: 861
Number of predicates: 8
Number of entities: 1508


In [235]:
get_all_local_props_query = f"""
SELECT DISTINCT ?p WHERE {{
    ?s ?p ?o
    FILTER(STRSTARTS(STR(?p), "{(str(EX_PROP))}"))
}}
ORDER BY ?p
"""

get_all_local_props_results = g.query(get_all_local_props_query)

local_props = [row.p for row in get_all_local_props_results]

print("Number of local properties:", len(local_props))

Number of local properties: 5


In [236]:
local_props

[rdflib.term.URIRef('http://www.example.org/football/prop/headCoach'),
 rdflib.term.URIRef('http://www.example.org/football/prop/locatedIn'),
 rdflib.term.URIRef('http://www.example.org/football/prop/nationality'),
 rdflib.term.URIRef('http://www.example.org/football/prop/playsFor'),
 rdflib.term.URIRef('http://www.example.org/football/prop/playsPosition')]

In [237]:
prop_alignement_graph = Graph()

In [238]:
for prop_uri in local_props:
    prop_alignement_graph.add((prop_uri, RDF.type, RDF.Property))

Function to find a property existing between two wiki entities

In [ ]:
def find_matching_properties(subject_id, object_id):
    """Find a property existing between two wiki entities"""
    query = f"""
    SELECT DISTINCT ?prop ?propLabel ?propDescription WHERE {{
      wd:{subject_id} ?p wd:{object_id} .
      ?prop wikibase:directClaim ?p .
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    """
    results = exec_wikidata_sparql(query)

    matches = []
    for result in results["results"]["bindings"]:
        p_id = result["prop"]["value"].split("/")[-1]
        p_label = result.get("propLabel", {}).get("value", "N/A")
        p_desc = result.get("propDescription", {}).get("value", "")
        matches.append((p_id, p_label, p_desc))

    return pd.DataFrame(matches, columns=["id", "label", "description"])

Predicate `playsFor`

In [ ]:
alignement_mapping_df.loc[
    alignement_mapping_df["Private Entity"] == str(EX.term("Bukayo_Saka"))
]

,Private Entity,External URI,Confidence
597,http://www.example.org/football/Bukayo_Saka,http://www.wikidata.org/entity/Q59306386,100.0


In [ ]:
alignement_mapping_df.loc[
    alignement_mapping_df["Private Entity"] == str(EX.term("Arsenal_FC"))
]

,Private Entity,External URI,Confidence
596,http://www.example.org/football/Arsenal_FC,http://www.wikidata.org/entity/Q9617,90.91


In [ ]:
find_matching_properties("Q59306386", "Q9617")

,id,label,description
0,P54,member of sports team,sports teams or clubs that the subject represe...


In [243]:
# playsFor equivalent property P54
prop_alignement_graph.add((EX_PROP.playsFor, OWL.equivalentProperty, WIKIDATA_NS.term("P54")))

<Graph identifier=N8cc4a8c22dfb4aeeaef83ce06fc5ec8b (<class 'rdflib.graph.Graph'>)>

Predicate `locatedIn`

In [ ]:
alignement_mapping_df.loc[
    alignement_mapping_df["Private Entity"] == str(EX.term("England"))
]

,Private Entity,External URI,Confidence
704,http://www.example.org/football/England,http://www.wikidata.org/entity/Q21,100.0


In [ ]:
find_matching_properties("Q9617", "Q21")

,id,label,description


It looks like no matching property is found between Arsenal and England equivalent entities on wikidata.

Let's try with a French club : `Paris Saint-Germain --(?p)--> France`

In [ ]:
search_wikidata_aligned_entities_sparql("Paris Saint Germain", confidence_fn=similarity_matcher_score)

,id,label,score,description
0,Q483020,Paris Saint-Germain FC,92.68,"association football club in Paris, France"


In [ ]:
search_wikidata_aligned_entities_sparql("France", confidence_fn=similarity_matcher_score)

,id,label,score,description
0,Q142,France,100.0,country in Western Europe and other continents...


In [ ]:
find_matching_properties("Q483020", "Q142")

,id,label,description
0,P17,country,sovereign state that this item is in (not to b...


So it seems like P17 is the equivalent property on wikidata. But toward which country it links Arsenal club.

In [ ]:
query = """
SELECT DISTINCT ?object ?objectLabel WHERE {
  BIND(wd:Q9617 AS ?subject) # Arsenal
  BIND(wdt:P17 AS ?prop) # country

  ?subject ?prop ?object

  SERVICE wikibase:label { bd:serviceParam wikibase:language "en,fr". }
}
"""

exec_wikidata_sparql(query)

{'head': {'vars': ['object', 'objectLabel']},
 'results': {'bindings': [{'object': {'type': 'uri',
     'value': 'http://www.wikidata.org/entity/Q145'},
    'objectLabel': {'xml:lang': 'en',
     'type': 'literal',
     'value': 'United Kingdom'}}]}}

So let's find which property exists between UK (Q145) and England (Q21)

In [ ]:
find_matching_properties("Q145", "Q21")

,id,label,description
0,P150,contains the administrative territorial entity,(list of) direct subdivisions of an administra...


And what's the symmetric

In [ ]:
find_matching_properties("Q21", "Q145")

,id,label,description
0,P131,located in the administrative territorial entity,the item is located on the territory of the fo...
1,P17,country,sovereign state that this item is in (not to b...


Let's consider P17 as a subproperty of our local property `locatedIn`. Meaning that anything associated to a country is located in this country.



In [244]:
prop_alignement_graph.add((WIKIDATA_NS.term("P17"), RDFS.subPropertyOf, EX_PROP.locatedIn))

<Graph identifier=N8cc4a8c22dfb4aeeaef83ce06fc5ec8b (<class 'rdflib.graph.Graph'>)>

Predicate `playsPosition`

In [ ]:
alignement_mapping_df.loc[
    alignement_mapping_df["Private Entity"] == str(EX.term("Goalkeeper"))
]

,Private Entity,External URI,Confidence
696,http://www.example.org/football/Goalkeeper,http://www.wikidata.org/entity/Q201330,100.0


In [ ]:
next(iter(g.subjects(EX_PROP.playsPosition, EX.term("Goalkeeper"))))

rdflib.term.URIRef('http://www.example.org/football/Fraser_Forster')

In [ ]:
alignement_mapping_df.loc[
    alignement_mapping_df["Private Entity"] == str(EX.term("Fraser_Forster"))
]

,Private Entity,External URI,Confidence
441,http://www.example.org/football/Fraser_Forster,http://www.wikidata.org/entity/Q348813,100.0


In [ ]:
find_matching_properties("Q348813", "Q201330")

,id,label,description
0,P413,position played on team / speciality,position or specialism of a player on a team


So P413 wil be considered as the equivalent for our local property `playsPosition`

In [245]:
prop_alignement_graph.add((EX_PROP.playsPosition, OWL.equivalentProperty, WIKIDATA_NS.term("P413")))

<Graph identifier=N8cc4a8c22dfb4aeeaef83ce06fc5ec8b (<class 'rdflib.graph.Graph'>)>

Predicate `nationality`

In [ ]:
alignement_mapping_df.loc[
    alignement_mapping_df["Private Entity"] == str(EX.term("Italy"))
]

,Private Entity,External URI,Confidence
641,http://www.example.org/football/Italy,http://www.wikidata.org/entity/Q38,100.0


In [ ]:
next(iter(g.subjects(EX_PROP.nationality, EX.term("Italy"))))

rdflib.term.URIRef('http://www.example.org/football/Gianluigi_Donnarumma')

In [ ]:
alignement_mapping_df.loc[
    alignement_mapping_df["Private Entity"] == str(EX.term("Gianluigi_Donnarumma"))
]

,Private Entity,External URI,Confidence
407,http://www.example.org/football/Gianluigi_Donn...,http://www.wikidata.org/entity/Q20830808,100.0


In [ ]:
find_matching_properties("Q20830808", "Q38")

,id,label,description
0,P27,country of citizenship,the object is a country that recognizes the su...
1,P1532,country for sport,country a person or a team represents when pla...


In [246]:
prop_alignement_graph.add((EX_PROP.nationality, OWL.equivalentProperty, WIKIDATA_NS.term("P27")))

<Graph identifier=N8cc4a8c22dfb4aeeaef83ce06fc5ec8b (<class 'rdflib.graph.Graph'>)>

Predicate `headCoach`

In [ ]:
alignement_mapping_df.loc[
    alignement_mapping_df["Private Entity"] == str(EX.term("Mikel_Arteta"))
]

,Private Entity,External URI,Confidence
707,http://www.example.org/football/Mikel_Arteta,http://www.wikidata.org/entity/Q185572,100.0


In [ ]:
find_matching_properties("Q9617", "Q185572")

,id,label,description
0,P286,head coach,on-field manager or head coach of a sports clu...


In [247]:
prop_alignement_graph.add((EX_PROP.headCoach, OWL.equivalentProperty, WIKIDATA_NS.term("P286")))

<Graph identifier=N8cc4a8c22dfb4aeeaef83ce06fc5ec8b (<class 'rdflib.graph.Graph'>)>

In [249]:
# for prop_triplet in prop_alignement_graph:
#     prop_labels_triplet = [
#         item.split("/")[-1]
#         for item in prop_triplet
#     ]
#     print(prop_labels_triplet)

Combine with orignal graph and save

In [223]:
aligned_graph_fpath = "aligned_football_kb.ttl"

In [251]:
aligned_graph = g + prop_alignement_graph

aligned_graph.serialize(destination=aligned_graph_fpath, format="turtle")

<Graph identifier=N4b5fae7dba8e45b99dbeccf7973b0bf3 (<class 'rdflib.graph.Graph'>)>

In [ ]:
len(aligned_graph)

# Step 4 --- KB Expansion via SPARQL

In [255]:
g = Graph().parse(aligned_graph_fpath)

In [256]:
display_graph_stats(g)

Number of triples: 3516
Number of subjects: 778
Number of objects: 867
Number of predicates: 10
Number of entities: 1519


In [257]:
def filter_properties(df, blacklist):
    """
    df: DataFrame with column 'prop_id'
    blacklist: list of properties to exclude (ex: ["P735", "P18"])
    """
    if not blacklist:
        return df

    df_filtered = df[~df["prop_id"].isin(blacklist)].copy()

    df_filtered.drop(columns=["prop_id"], inplace=True)

    return df_filtered

In [258]:
def get_wiki_properties_query(wiki_ids, as_subject=True, limit=100, blacklist=[]):
    entities_candidates = " ".join([f"wd:{wiki_id}" for wiki_id in wiki_ids])
    blacklist_candidates = ",".join([f"wdt:{prop_id}" for prop_id in blacklist])

    id_as_subject_selector = "?x ?p ?y"
    id_as_object_selector = "?y ?p ?x"

    prop_selector = id_as_subject_selector if as_subject else id_as_object_selector
    query = f"""
    SELECT ?prop ?propLabel ?propDescription (COUNT(*) as ?count)
    WHERE {{
        VALUES ?x {{
            {entities_candidates}
        }}
        {prop_selector} .
        ?prop wikibase:directClaim ?p .
        FILTER(
            ?p NOT IN ({blacklist_candidates}) &&
            isIRI(?y) &&
            STRSTARTS(STR(?y), "http://www.wikidata.org/entity/Q")
        )
        SERVICE wikibase:label {{
            bd:serviceParam wikibase:language "en".
        }}

    }}
    # LIMIT {limit}
    GROUP BY ?prop ?propLabel ?propDescription
    ORDER BY DESC(?count)
    LIMIT {limit}
    """.strip()
    return query



def get_wiki_properties(wiki_ids, as_subject=True, limit=100, blacklist=[]):
    """
    Collect a list of properties and order them by the frequency of apparition.
    Apply properties filtering if blacklist is not empty.
    """
    query = get_wiki_properties_query(wiki_ids, as_subject=as_subject, limit=limit, blacklist=blacklist)

    results = exec_wikidata_sparql(query)

    data = []
    for result in results["results"]["bindings"]:
        prop = result["prop"]["value"]
        count = int(result["count"]["value"])
        label = result.get("propLabel", {}).get("value", "")
        description = result.get("propDescription", {}).get("value", "")

        data.append({
            "prop_id": prop.split("/")[-1],
            "prop": prop,
            "label": label,
            "description": description,
            "count": count
        })

    df = pd.DataFrame(data)
    return df

Get a list of wiki ids for local teams

In [259]:
get_team_wiki_ids_query = f"""
PREFIX : <{EX}>
PREFIX owl: <{OWL}>

SELECT ?team_wiki_id
WHERE {{
    ?team_local_id rdf:type :Team .
    ?team_local_id owl:sameAs ?team_wiki_id
}}
"""

team_wiki_uris = list(g.query(get_team_wiki_ids_query))
team_wiki_ids = [uri.split("/")[-1] for (uri,) in team_wiki_uris]

print("Total Entities:", len(team_wiki_ids))
team_wiki_ids[:5]

Total Entities: 20


['Q18708', 'Q1128631', 'Q18747', 'Q19500', 'Q19568']

Get properties with their frequencies

In [260]:
team_prop_blacklist = [
    "P31", # instance type
    "P54", # plays for
    "P286", # head coach
]
team_prop_df = get_wiki_properties(
    team_wiki_ids[:20],
    limit=200,
    blacklist=team_prop_blacklist,
)

# team_as_object_prop_df = get_wiki_properties(
#     team_wiki_ids[:20],
#     as_subject=False,
#     limit=200,
#     blacklist=team_prop_blacklist,
# )
# team_prop_df = pd.concat(
#     (team_prop_df, team_as_object_prop_df),
#     ignore_index=True
# ).sort_values(by="count", ascending=False)

team_prop_df = team_prop_df.loc[team_prop_df["count"] >= 3]

print("Shape:", team_prop_df.shape)
team_prop_df.head()

Shape: (24, 5)


,prop_id,prop,label,description,count
0,P1344,http://www.wikidata.org/entity/P1344,participant in,"event in which a person, organization or creat...",92
1,P2522,http://www.wikidata.org/entity/P2522,competition won,competition or sports event won by the subject,42
2,P1424,http://www.wikidata.org/entity/P1424,topic has template,the main template relating to a topic,41
3,P6364,http://www.wikidata.org/entity/P6364,official color,official colors chosen to represent or identif...,40
4,P3279,http://www.wikidata.org/entity/P3279,statistical leader,leader of a sports tournament in one of statis...,34


Do the same with players

In [261]:
get_people_wiki_ids_query = f"""
PREFIX : <{EX}>
PREFIX owl: <{OWL}>

SELECT ?people_wiki_id
WHERE {{
    ?people_local_id rdf:type :Person .
    ?people_local_id owl:sameAs ?people_wiki_id
}}
"""

people_wiki_uris = list(g.query(get_people_wiki_ids_query))
people_wiki_ids = [uri.split("/")[-1] for (uri,) in people_wiki_uris]
print("Total Entites:", len(people_wiki_ids))

people_prop_blacklist = [
    "P31", # instance type
    "P54", # plays for
    "P413", # plays position
    "P286", # head coach
    "P27", # nationality
    "P735", # links player to an entity URI associated to their firstname
    "P734", # links player to an entity URI associated to their lastname
]
people_prop_df = get_wiki_properties(
    people_wiki_ids[:20],
    limit=200,
    blacklist=people_prop_blacklist,
)
people_as_object_prop_df = get_wiki_properties(
    people_wiki_ids[:20],
    as_subject=False,
    limit=200,
    blacklist=people_prop_blacklist,
)
people_prop_df = pd.concat(
    (people_prop_df, people_as_object_prop_df),
    ignore_index=True
).sort_values(by="count", ascending=False)

people_prop_df = people_prop_df.loc[people_prop_df["count"] >= 3]

print(people_prop_df.shape)
people_prop_df.head()

Total Entites: 629
(12, 5)


,prop_id,prop,label,description,count
26,P710,http://www.wikidata.org/entity/P710,participant,"person, group of people or organization (objec...",49
0,P1344,http://www.wikidata.org/entity/P1344,participant in,"event in which a person, organization or creat...",29
1,P106,http://www.wikidata.org/entity/P106,occupation,"occupation of a person. See also ""field of wor...",24
2,P21,http://www.wikidata.org/entity/P21,sex or gender,sex or gender identity of human or animal. For...,20
3,P641,http://www.wikidata.org/entity/P641,sport,sport that the subject participates or partici...,17


In [262]:
get_country_wiki_ids_query = f"""
PREFIX : <{EX}>
PREFIX owl: <{OWL}>

SELECT ?country_wiki_id
WHERE {{
    ?country_local_id rdf:type :Country .
    ?country_local_id owl:sameAs ?country_wiki_id
}}
"""

country_wiki_uris = list(g.query(get_country_wiki_ids_query))
country_wiki_ids = [uri.split("/")[-1] for (uri,) in country_wiki_uris]
print("Total Entities:", len(country_wiki_ids))

country_prop_blacklist = [
    "P31", # instance type
    "P27", # nationality
    "P17", # sovereign state that this item is in (heavy)
]
country_prop_df = get_wiki_properties(
    country_wiki_ids[:3],
    limit=200,
    blacklist=country_prop_blacklist,
)
# country_as_object_prop_df = get_wiki_properties(
#     country_wiki_ids[:20],
#     as_subject=False,
#     limit=200,
#     blacklist=country_prop_blacklist,
# )
# country_prop_df = pd.concat(
#     (country_prop_df, country_as_object_prop_df),
#     ignore_index=True
# ).sort_values(by="count", ascending=False)

country_prop_df = country_prop_df.loc[country_prop_df["count"] >= 3]

print(country_prop_df.shape)
country_prop_df.head()

Total Entities: 68
(52, 5)


,prop_id,prop,label,description,count
0,P530,http://www.wikidata.org/entity/P530,diplomatic relation,diplomatic relations of the country,95
1,P150,http://www.wikidata.org/entity/P150,contains the administrative territorial entity,(list of) direct subdivisions of an administra...,81
2,P1343,http://www.wikidata.org/entity/P1343,described by source,"work where this item is described, in statisti...",48
3,P463,http://www.wikidata.org/entity/P463,member of,"organization, club or musical group to which t...",32
4,P2936,http://www.wikidata.org/entity/P2936,language used,language widely used (spoken or written) in th...,28


In [263]:
team_prop_ids = team_prop_df["prop_id"].iloc[:50].tolist()
people_prop_ids = people_prop_df["prop_id"].iloc[:50].tolist()
country_prop_ids = country_prop_df["prop_id"].iloc[:50].tolist()

In [264]:
get_remaining_wiki_ids_query = f"""
PREFIX : <{EX}>
PREFIX owl: <{OWL}>

SELECT ?remaining_wiki_id
WHERE {{
    ?remaining_local_id rdf:type ?local_type .
    ?remaining_local_id owl:sameAs ?remaining_wiki_id

    FILTER(?local_type != :Team && ?local_type != :Person && ?local_type != :Country)
}}
"""

remaining_wiki_uris = list(g.query(get_remaining_wiki_ids_query))
remaining_wiki_ids = [uri.split("/")[-1] for (uri,) in remaining_wiki_uris]

print("Total Entities:", len(remaining_wiki_ids))
remaining_wiki_ids[:5]

Total Entities: 16


['Q861529', 'Q114358125', 'Q114358123', 'Q66115689', 'Q90326494']

In [265]:
def get_wiki_expansion_triplets(
    wiki_ids,
    prop_ids=[],
    prop_min_freq=0,
    limit=1000,
):
    """
    Build a list containing triplets for a single hop expansion based on a list of wiki entities.
    """
    # Build a list of properties (appearing only once) to take into account if `unique` is passed
    if prop_ids == "unique":
        prop_df = get_wiki_properties(wiki_ids[:20])
        prop_df = prop_df.loc[prop_df["count"] >= prop_min_freq]

        prop_ids = prop_df["prop_id"].tolist()


    entities_candidates = "\n".join([f"wd:{wiki_id}" for wiki_id in wiki_ids])

    # query without properties filtering
    query = f"""
    SELECT ?subject ?prop ?object
    WHERE {{
        VALUES ?x {{
            {entities_candidates}
        }}

        {{
            ?x ?p ?object .
            ?prop wikibase:directClaim ?p .
            BIND(?x AS ?subject)
            FILTER(
              isIRI(?object) &&
              STRSTARTS(STR(?object), "http://www.wikidata.org/entity/Q")
            )
        }}

        UNION

        {{
            ?subject ?p ?x .
            ?prop wikibase:directClaim ?p .
            BIND(?x AS ?object)
            FILTER(
              isIRI(?subject) &&
              STRSTARTS(STR(?subject), "http://www.wikidata.org/entity/Q")
            )
        }}
    }}
    LIMIT {limit}
    """.strip()

    # query with properties filtering
    if prop_ids:
        prop_candidates = "\n".join([f"wdt:{prop_id}" for prop_id in prop_ids])
        query = f"""
        SELECT ?subject ?prop ?object
        WHERE {{
            VALUES ?x {{
                {entities_candidates}
            }}

            VALUES ?prop {{
                {prop_candidates}
            }}

            {{
                ?x ?prop ?object .
                BIND(?x AS ?subject)
                FILTER(
                  isIRI(?object) &&
                  STRSTARTS(STR(?object), "http://www.wikidata.org/entity/Q")
                )
            }}

            UNION

            {{
                ?subject ?prop ?x .
                BIND(?x AS ?object)
                FILTER(
                  isIRI(?subject) &&
                  STRSTARTS(STR(?subject), "http://www.wikidata.org/entity/Q")
                )
            }}
        }}
        LIMIT {limit}
        """.strip()

    # Execute query
    results = exec_wikidata_sparql(query)

    expansion_triplets = []
    if not results or not results.get("results") or not results["results"].get("bindings"):
        return expansion_triplets

    # Assemble results
    for result in results["results"]["bindings"]:
        subject = result["subject"]["value"]
        prop = result["prop"]["value"]
        obj = result["object"]["value"]
        expansion_triplets.append((URIRef(subject), URIRef(prop), URIRef(obj)))

    return expansion_triplets

In [266]:
def extend_graph(graph, triplets):
    for triplet in triplets:
        graph.add(triplet)
    return graph

1 hop expansion

In [268]:
batch_size = 10

start_time = time.perf_counter()
expansion_triplets = []

print("Expansion based on: Teams")
for i in range(0, len(team_wiki_ids), batch_size):
    print("Batch", i//batch_size + 1, end="")
    batch_ids = team_wiki_ids[i:i+batch_size]
    exp_triplets = get_wiki_expansion_triplets(batch_ids, prop_ids=team_prop_ids, limit=1000)
    expansion_triplets.extend(exp_triplets)
    print("... Done")

print()
print("Expansion based on: People")
for i in range(0, len(people_wiki_ids), batch_size):
    print("Batch", i//batch_size + 1, end="")
    batch_ids = people_wiki_ids[i:i+batch_size]
    exp_triplets = get_wiki_expansion_triplets(batch_ids, prop_ids=people_prop_ids, limit=1000)
    expansion_triplets.extend(exp_triplets)
    print("... Done")

print()
print("Expansion based on: Country")
for i in range(0, len(country_wiki_ids), batch_size):
    print("Batch", i//batch_size + 1, end="")
    batch_ids = country_wiki_ids[i:i+batch_size]
    exp_triplets = get_wiki_expansion_triplets(batch_ids, prop_ids=country_prop_ids, limit=1000)
    expansion_triplets.extend(exp_triplets)
    print("... Done")

print()
print("Expansion based on: Remaining")
for i in range(0, len(remaining_wiki_ids), batch_size):
    print("Batch", i//batch_size + 1, end="")
    batch_ids = remaining_wiki_ids[i:i+batch_size]
    exp_triplets = get_wiki_expansion_triplets(batch_ids, prop_ids="unique", limit=1000)
    expansion_triplets.extend(exp_triplets)
    print("... Done")

end_time = time.perf_counter()
print()
print(f"Time taken: {end_time - start_time:0.4f} seconds")
print(f"{len(expansion_triplets)} Triplets obtained")

Expansion based on: Teams
Batch 1... Done
Batch 2... Done

Expansion based on: People
Batch 1... Done
Batch 2... Done
Batch 3... Done
Batch 4... Done
Batch 5... Done
Batch 6... Done
Batch 7... Done
Batch 8... Done
Batch 9... Done
Batch 10... Done
Batch 11... Done
Batch 12... Done
Batch 13... Done
Batch 14... Done
Batch 15... Done
Batch 16... Done
Batch 17... Done
Batch 18... Done
Batch 19... Done
Batch 20... Done
Batch 21... Done
Batch 22... Done
Batch 23... Done
Batch 24... Done
Batch 25... Done
Batch 26... Done
Batch 27... Done
Batch 28... Done
Batch 29... Done
Batch 30... Done
Batch 31... Done
Batch 32... Done
Batch 33... Done
Batch 34... Done
Batch 35... Done
Batch 36... Done
Batch 37... Done
Batch 38... Done
Batch 39... Done
Batch 40... Done
Batch 41... Done
Batch 42... Done
Batch 43... Done
Batch 44... Done
Batch 45... Done
Batch 46... Done
Batch 47... Done
Batch 48... Done
Batch 49... Done
Batch 50... Done
Batch 51... Done
Batch 52... Done
Batch 53... Done
Batch 54... Done
Batch

In [269]:
first_hop_expansion_graph = Graph()

first_hop_expansion_graph = extend_graph(first_hop_expansion_graph, expansion_triplets)
display_graph_stats(first_hop_expansion_graph)

Number of triples: 18936
Number of subjects: 4011
Number of objects: 4662
Number of predicates: 99
Number of entities: 8035


In [270]:
known_wiki_ids = team_wiki_ids + people_wiki_ids + country_wiki_ids + remaining_wiki_ids

In [271]:
first_hop_obj_uris = list(first_hop_expansion_graph.objects(None, None))

first_hop_obj_ids = [
    uri.split("/")[-1]
    for uri in first_hop_obj_uris
    if str(WIKIDATA_NS) in str(uri)
]

first_hop_obj_ids = list(set(first_hop_obj_ids))

new_hop_ids = [
    ent_id
    for ent_id in first_hop_obj_ids
    if ent_id not in known_wiki_ids
]

len(new_hop_ids)

4256

In [272]:
batch_size = 20

start_time = time.perf_counter()
expansion_triplets = []
for i in range(0, len(new_hop_ids), batch_size):
    print("Batch", i//batch_size + 1, end="")
    batch_ids = new_hop_ids[i:i+batch_size]
    exp_triplets = get_wiki_expansion_triplets(batch_ids, limit=200)
    expansion_triplets.extend(exp_triplets)
    print("... Done")

end_time = time.perf_counter()
print()
print(f"Time taken: {end_time - start_time:0.4f} seconds")
print(f"{len(expansion_triplets)} Triplets obtained")

end_time = time.perf_counter()

Batch 1... Done
Batch 2... Done
Batch 3... Done
Batch 4... Done
Batch 5... Done
Batch 6... Done
Batch 7... Done
Batch 8... Done
Batch 9... Done
Batch 10... Done
Batch 11... Done
Batch 12... Done
Batch 13... Done
Batch 14... Done
Batch 15... Done
Batch 16... Done
Batch 17... Done
Batch 18... Done
Batch 19... Done
Batch 20... Done
Batch 21... Done
Batch 22... Done
Batch 23... Done
Batch 24... Done
Batch 25... Done
Batch 26... Done
Batch 27... Done
Batch 28... Done
Batch 29... Done
Batch 30... Done
Batch 31... Done
Batch 32... Done
Batch 33... Done
Batch 34... Done
Batch 35... Done
Batch 36... Done
Batch 37... Done
Batch 38... Done
Batch 39... Done
Batch 40... Done
Batch 41... Done
Batch 42... Done
Batch 43... Done
Batch 44... Done
Batch 45... Done
Batch 46... Done
Batch 47... Done
Batch 48... Done
Batch 49... Done
Batch 50... Done
Batch 51... Done
Batch 52... Done
Batch 53... Done
Batch 54... Done
Batch 55... Done
Batch 56... Done
Batch 57... Done
Batch 58... Done
Batch 59... Done
Batch 

In [273]:
new_hop_expansion_graph = Graph()

new_hop_expansion_graph = extend_graph(new_hop_expansion_graph, expansion_triplets)
display_graph_stats(new_hop_expansion_graph)

Number of triples: 42458
Number of subjects: 14801
Number of objects: 17839
Number of predicates: 350
Number of entities: 31720


In [274]:
expanded_graph = g + first_hop_expansion_graph + new_hop_expansion_graph

display_graph_stats(expanded_graph)

Number of triples: 64910
Number of subjects: 19382
Number of objects: 21445
Number of predicates: 459
Number of entities: 38655


In [275]:
expanded_graph_fpath = "expanded_football_kb.ttl"

In [276]:
expanded_graph.serialize(destination=expanded_graph_fpath, format="turtle")

<Graph identifier=N301b5227ce8a4d849187eef728467db1 (<class 'rdflib.graph.Graph'>)>

## Cleaning Before Embedding

In [277]:
from collections import defaultdict

In [278]:
g = Graph().parse(expanded_graph_fpath)

URI consistency

In [279]:
uri_consistency_selector = """
WHERE {{
    ?s ?p ?o .
    FILTER(!isIRI(?s) || !isIRI(?p) || !isIRI(?o))
}}
""".strip()

check_uri_consistency_query = f"""
SELECT ?s ?p ?o
{uri_consistency_selector}
""".strip()

check_uri_consistency_results = g.query(check_uri_consistency_query)

result_iter = iter(check_uri_consistency_results)
for _ in range(3):
    try:
        row = next(result_iter)
    except StopIteration:
        print("Nothing more to see")
        break
    print(row)

Nothing more to see


In [280]:
# delete_uri_consistency_query = f"""
# DELETE {{
#     ?s ?p ?o
# }}
# {uri_consistency_selector}
# """.strip()

# delete_uri_consistency_results = g.update(delete_uri_consistency_query)

In [281]:
outer_namespaces_entities_selector = f"""
WHERE {{
    ?s ?p ?o .

    FILTER(
        (
            !STRSTARTS(STR(?s), "{EX}") &&
            !STRSTARTS(STR(?s), "{WIKIDATA_BASE}")
        )
        ||
        (
            !STRSTARTS(STR(?p), "{EX}") &&
            !STRSTARTS(STR(?p), "{WIKIDATA_BASE}") &&
            !STRSTARTS(STR(?p), "{RDF}") &&
            !STRSTARTS(STR(?p), "{RDFS}") &&
            !STRSTARTS(STR(?p), "{OWL}")
        )
        ||
        (
            isIRI(?o) &&
            !STRSTARTS(STR(?o), "{EX}") &&
            !STRSTARTS(STR(?o), "{WIKIDATA_BASE}") &&
            !STRSTARTS(STR(?o), "{RDF}") &&
            !STRSTARTS(STR(?o), "{RDFS}")
        )
    )
}}
""".strip()

check_outer_namespaces_entities_query = f"""
SELECT ?s ?p ?o
{outer_namespaces_entities_selector}
""".strip()

check_outer_namespaces_entities_results = g.query(check_outer_namespaces_entities_query)

result_iter = iter(check_outer_namespaces_entities_results)
for i in range(3):
    try:
        row = next(result_iter)
    except StopIteration:
        print("Nothing more to see")
        break
    print(row)

Nothing more to see


In [282]:
# delete_outer_namespaces_entities_query = f"""
# DELETE {{
#     ?s ?p ?o
# }}
# {outer_namespaces_entities_selector}
# """.strip()

# delete_outer_namespaces_entities_results = g.update(delete_outer_namespaces_entities_query)

In [283]:
display_graph_stats(g)

Number of triples: 64910
Number of subjects: 19382
Number of objects: 21445
Number of predicates: 459
Number of entities: 38655


In [284]:
def get_isolated_triplets(graph):
    degree_map = defaultdict(int)

    for s, p, o in graph:
        degree_map[s] += 1
        degree_map[o] += 1

    unique_degree_nodes = {node for node, d in degree_map.items() if d <= 1}

    isolated_triplets = []
    for s, p, o in list(graph):
        if s in unique_degree_nodes and o in unique_degree_nodes:
            isolated_triplets.append((s, p, o))
    return isolated_triplets

In [285]:
isolated_triplets = get_isolated_triplets(g)

len(isolated_triplets)

0

In [286]:
isolated_triplets

[]

In [287]:
# for triplet in isolated_triplets:
#     g.remove(triplet)

# display_graph_stats(g)

In [288]:
clean_graph_fpath = "clean_football_kb.ttl"

In [289]:
g.serialize(destination=clean_graph_fpath, format="turtle")

<Graph identifier=Nd582dcfaaa2147f7856192112df2d188 (<class 'rdflib.graph.Graph'>)>